# Chat com PDF - IA Generativa e RAG

> **Objetivo:** Construir um assistente inteligente capaz de responder dúvidas técnicas sobre documentos internos em formato PDF, mitigando alucinações ao recuperar os trechos relevantes antes de invocar o LLM e citando a numeração exata das páginas.

### Tecnologias:
- **LLM:** Google Gemini (`gemini-3.5-flash-lite` via SDK oficial `google-genai`)
- **Extração de Texto:** `PyPDF2`
- **Chunking & Indexação:** Divisão por caracteres com sobreposição (*overlap*) e recuperação textual via **TF-IDF** + **Similaridade de Cosseno** (`scikit-learn`)
- **Interface de Usuário:** `Gradio` (Chat web interativo com link público)

---
### Fluxo da solucao:
```
[PDF] -> [PyPDF2: Extração por Página] -> [Chunking com Overlap]
                                                |
                                                v
[Pergunta do Usuário] -> [Busca TF-IDF] -> [Trechos Relevantes + Páginas]
                                                |
                                                v
                                 [Prompt Contextualizado + Regras]
                                                |
                                                v
                                   [Gemini API -> Resposta com Fontes]
```

## 1. Instalação das Dependências
Instalamos as bibliotecas necessárias para extração de PDF, busca vetorial/estatística, interface web e o SDK oficial do Google GenAI.

In [ ]:
%%capture
# Instalacao silenciosa dos pacotes necessarios
!pip install -q google-genai PyPDF2 scikit-learn gradio

## 2. Configuração do Cliente Gemini
Autenticação com a API do Google Gemini. O código recupera a chave de forma segura:
1. Via **Secrets do Google Colab** (`userdata.get('GEMINI_API_KEY')`), ou
2. Via **Variável de Ambiente** (`os.environ['GEMINI_API_KEY']`) para execução local.

In [ ]:
import os
import json
from google import genai

# Tenta recuperar dos Secrets do Colab, depois do ambiente local
api_key = None
try:
    from google.colab import userdata
    api_key = userdata.get("GEMINI_API_KEY")
except (ImportError, Exception):
    api_key = os.environ.get("GEMINI_API_KEY")

# Trata caso o segredo tenha sido salvo como dicionário ou JSON
if isinstance(api_key, dict):
    for k in ["GEMINI_API_KEY", "api_key", "key", "token"]:
        if k in api_key and isinstance(api_key[k], str):
            api_key = api_key[k]
            break
    else:
        for v in api_key.values():
            if isinstance(v, str):
                api_key = v
                break

if isinstance(api_key, str):
    api_key = api_key.strip()
    if api_key.startswith("{") and api_key.endswith("}"):
        try:
            dados = json.loads(api_key)
            if isinstance(dados, dict):
                for k in ["GEMINI_API_KEY", "api_key", "key", "token"]:
                    if k in dados and isinstance(dados[k], str):
                        api_key = dados[k].strip()
                        break
                else:
                    for v in dados.values():
                        if isinstance(v, str):
                            api_key = v.strip()
                            break
        except Exception:
            pass

# Caso ainda não encontre ou não seja string válida, solicita interativamente
if not api_key or not isinstance(api_key, str):
    import getpass
    api_key = getpass.getpass("Digite sua GEMINI_API_KEY (string pura, ex: AIzaSy...): ").strip()

# Inicializa o cliente oficial do Google GenAI
client = genai.Client(api_key=api_key)
print("Cliente Gemini configurado com sucesso.")

## 3. Upload e Extração de Texto do PDF
Utilizamos a biblioteca `PyPDF2` para percorrer cada página do documento. Preservamos o número da página junto ao texto extraído para permitir a citação precisa de fontes na resposta final.

In [ ]:
import os
from PyPDF2 import PdfReader

# Procura por arquivos PDF locais (ex: sample.pdf) ou abre o diálogo de upload no Colab
pdf_candidatos = [f for f in os.listdir(".") if f.lower().endswith(".pdf")]

if pdf_candidatos:
    # Prioriza sample.pdf ou o primeiro encontrado
    nome_arquivo = "sample.pdf" if "sample.pdf" in pdf_candidatos else pdf_candidatos[0]
    print(f"Carregando documento existente: {nome_arquivo}")
else:
    from google.colab import files
    print("Por favor, faça o upload do seu arquivo PDF:")
    uploaded = files.upload()
    nome_arquivo = next(iter(uploaded))

# Leitura do PDF e extração de páginas
reader = PdfReader(nome_arquivo)
paginas = []

for numero, pagina in enumerate(reader.pages, start=1):
    texto = pagina.extract_text() or ""
    paginas.append({
        "pagina": numero,
        "texto": texto.strip()
    })

total_com_conteudo = sum(bool(p["texto"]) for p in paginas)
print(f"Arquivo carregado: {nome_arquivo}")
print(f"Total de paginas: {len(paginas)}")
print(f"Paginas com texto: {total_com_conteudo}")

## 4. Chunking (Divisão em Trechos com Sobreposição)
Modelos de linguagem possuem melhor precisão quando fornecemos trechos específicos e delimitados. Além disso:
- **`tamanho` (800 caracteres):** Tamanho ideal para conter uma ideia ou parágrafo completo.
- **`overlap` (150 caracteres):** Garante que frases ou conceitos cortados na fronteira entre blocos não percam o sentido.

In [ ]:
def criar_chunks(paginas, tamanho=800, overlap=150):
    """
    Divide o texto das páginas em blocos menores com sobreposição (overlap).
    Mantém a referência da página de origem em cada bloco.
    """
    chunks = []
    passo = tamanho - overlap

    for pagina in paginas:
        # Normaliza quebras de linha e espaços múltiplos em um texto corrido
        texto_normalizado = " ".join(pagina["texto"].split())

        if not texto_normalizado:
            continue

        for inicio in range(0, len(texto_normalizado), passo):
            trecho = texto_normalizado[inicio:inicio + tamanho]
            if trecho.strip():
                chunks.append({
                    "pagina": pagina["pagina"],
                    "texto": trecho
                })
    return chunks

chunks = criar_chunks(paginas)
print(f"Total de chunks: {len(chunks)}")
if chunks:
    print(f"Prévia do Chunk 1 (Pág. {chunks[0]['pagina']}):\n\"{chunks[0]['texto'][:160]}...\"")

## 5. Indexação Textual & Busca por Similaridade (TF-IDF)
Para recuperar os blocos relevantes sem necessidade de um banco vetorial externo pesado:
- Criamos a matriz de frequência de termos com o **`TfidfVectorizer`** (ponderando unigramas e bigramas);
- A função `buscar_chunks` calcula a **similaridade de cosseno** entre o vetor da pergunta e todos os chunks indexados, retornando os `top_k` trechos com maior afinidade temática.

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Selecionar páginas com conteúdo técnico (exclui capa, sumário e divisores de capítulo)
# No eBook Delta Lake: 1 (capa), 2 (sumário), 4, 10, 15 (títulos de capítulo) e 19 (contracapa)
PAGINAS_EXCLUIDAS = {1, 2, 4, 10, 15, 19}
chunks_busca = [
    chunk for chunk in chunks
    if chunk["pagina"] not in PAGINAS_EXCLUIDAS and len(chunk["texto"].split()) > 15
]

# Caso seja outro documento em que a exclusão não se aplique
if not chunks_busca:
    chunks_busca = [chunk for chunk in chunks if len(chunk["texto"].split()) > 15]

# Criação do vetor TF-IDF (unigramas e bigramas com ponderação sublinear)
vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    sublinear_tf=True
)

textos = [chunk["texto"] for chunk in chunks_busca]
matriz = vectorizer.fit_transform(textos)
print(f"Chunks indexados: {len(chunks_busca)}")

# Função de busca por similaridade de cosseno
def buscar_chunks(pergunta, limite=5):
    """Recupera os k chunks mais relevantes para a pergunta."""
    vetor = vectorizer.transform([pergunta])
    scores = cosine_similarity(vetor, matriz).ravel()
    indices = np.argsort(scores)[::-1][:limite]

    return [
        {
            **chunks_busca[int(i)],
            "similaridade": round(float(scores[i]), 4)
        }
        for i in indices
        if scores[i] > 0
    ]

print("Funcao de busca configurada.")

## 6. Pipeline RAG & Síntese com Google Gemini
A função `perguntar_pdf` orquestra a geração aumentada por recuperação:
1. Busca os trechos mais relevantes via TF-IDF;
2. Constrói o contexto com os trechos e a numeração das páginas;
3. Envia para o modelo (`gemini-3.5-flash-lite`) com instruções claras para:
   - **Não alucinar:** utilizar apenas o contexto fornecido;
   - **Citar fontes:** indicar expressamente o número da página;
   - **Fallback:** caso o assunto não esteja no contexto, responder exatamente: `"Não encontrei essa informação no documento."`.

In [ ]:
def perguntar_pdf(pergunta, modelo="gemini-3.5-flash-lite", limite=5):
    """
    Executa o pipeline RAG: busca trechos relevantes e sintetiza resposta detalhada com Gemini.
    """
    # Recuperar os trechos mais relevantes
    resultados = buscar_chunks(pergunta, limite=limite)

    if not resultados:
        return "Não encontrei essa informação no documento."

    # Montar o contexto para o Gemini
    contexto = "\n\n".join(
        f"[Página {r['pagina']}]\n{r['texto']}"
        for r in resultados
    )

    # Prompt com regras anti-alucinação e citação de páginas
    prompt = f"""
    Você é um assistente que responde perguntas
    exclusivamente com base em documentos.

    Regras:
    - Utilize somente as informações do contexto.
    - Não invente informações.
    - Cite as páginas utilizadas.
    - Responda em português de forma clara, completa e detalhada.
    - Caso não encontre a resposta, diga estritamente:
      "Não encontrei essa informação no documento."

    CONTEXTO:
    {contexto}

    PERGUNTA:
    {pergunta}
    """

    # Chamar a API do Gemini
    resposta = client.models.generate_content(
        model=modelo,
        contents=prompt
    )

    return resposta.text.strip()

print("Pipeline pronto.")

## 7. Validação do Desafio Prático (Critérios de Aceite)
Para atender aos entregáveis solicitados:
1. **3 Perguntas sobre o tema do documento**, com resposta precisa e citação das páginas.
2. **1 Pergunta fora do escopo do documento**, na qual o modelo responde categoricamente que a informação não foi encontrada.

In [ ]:
testes = [
    {
        "tipo": "Dentro do Escopo (Pergunta 1)",
        "pergunta": "What is Delta Lake and how does it bring reliability to data lakes?"
    },
    {
        "tipo": "Dentro do Escopo (Pergunta 2)",
        "pergunta": "What is a Lakehouse architecture and what problems does it solve?"
    },
    {
        "tipo": "Dentro do Escopo (Pergunta 3)",
        "pergunta": "How does Delta Lake handle ACID transactions on cloud object storage?"
    },
    {
        "tipo": "Fora do Escopo (Teste Anti-Alucinação)",
        "pergunta": "Qual foi a escalação da seleção brasileira na final da Copa do Mundo de 2002?"
    }
]

print("=" * 70)
print("BATERIA DE TESTES DE VALIDAÇÃO DO RAG")
print("=" * 70)

for t in testes:
    print(f"\n [{t['tipo']}]")
    print(f"Pergunta: {t['pergunta']}")
    resposta = perguntar_pdf(t["pergunta"])
    print(f"\nResposta:\n{resposta}")
    print("-" * 70)

## 8. Interface Interativa com Gradio
Para transformar o notebook em uma aplicação visual para o usuário final, utilizamos o `Gradio`.
O parâmetro `share=True` gera uma URL pública temporária (`.gradio.live`) que pode ser aberta em qualquer navegador ou celular.

In [ ]:
import gradio as gr

def responder_chat(mensagem, historico):
    """Conecta o input do chat ao pipeline RAG."""
    if not mensagem.strip():
        return "Por favor, digite uma pergunta válida."
    return perguntar_pdf(mensagem)

# Criação da interface interativa
demo = gr.ChatInterface(
    fn=responder_chat,
    title="Chat com PDF - Assistente RAG",
    description=(
        "Faça perguntas sobre o documento carregado. "
        "O assistente recupera os trechos mais relevantes do PDF e sintetiza a resposta "
        "com citação das páginas de referência."
    ),
    examples=[
        "What is a Lakehouse?",
        "What are the benefits of Delta Lake?",
        "How does ACID storage work on object stores?",
        "Quem foi o campeão da Copa de 2002?"
    ]
)

# Inicia o servidor local e gera link público temporário
demo.launch(share=True, debug=False)